# 04. Очистка с контролем качества — разбор и рабочие примеры

**Цель:** не написать максимально короткий `dropna()`, а сделать операции объяснимыми. Файл специально содержит противоречивые записи и неочевидные значения. Код здесь **показывает приёмы**, но не выполняет всю лабораторную за вас. Полный результат и журнал решений вы создаёте самостоятельно.

Среда: Python 3.10+, pandas. Запускайте notebook из папки `06_JUPYTER` (файлы данных расположены уровнем выше).

In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime
DATA = Path('../05_ДАННЫЕ_СТУДЕНТАМ')
raw = pd.read_csv(DATA/'campus_readings_RAW.csv', dtype='string', keep_default_na=False)
rooms = pd.read_csv(DATA/'rooms_catalog.csv', dtype='string')
print('наблюдений в файле:', len(raw), 'уникальных ID:', raw.record_id.nunique())
raw.head(5)

## 1. Зафиксировать состояние до очистки

Нельзя воспроизвести решение «стало лучше», если не измерено исходное состояние. Для текстовых столбцов пустой строкой может обозначаться пропуск; различайте пустоту и «0».

In [ ]:
missing = (raw[['co2_ppm','temp_c','people_count','observed_at']].apply(lambda col:col.str.strip().eq(''))).sum()
print('Пустые строки по столбцам:')
print(missing)
print('Полные дубли:',raw.duplicated().sum())
print('ID, встречающиеся более одного раза:')
print(raw.record_id.value_counts().loc[lambda s:s>1])

## 2. Точные дубли ≠ совпавшие числа

Внимательно проверьте: дубли строк; повторяющиеся технические ID; повторяющиеся пары (`co2_ppm`, `temp_c`). Три разных понятия — три разных проверки.

In [ ]:
exact_dups = raw.loc[raw.duplicated(keep=False)].sort_values('record_id')
id_collisions = raw.loc[raw.duplicated('record_id',keep=False)].sort_values('record_id')
print('строк в группах полных дублей:',len(exact_dups))
print('строк с повторяющимися ID:',len(id_collisions))
print(id_collisions[['record_id','observed_at','co2_ppm','temp_c']].head(12).to_string(index=False))

**Задание:** определите, какие пары отличаются содержательно. Нельзя автоматически применять `drop_duplicates(subset=['co2_ppm'])`. Сформулируйте основание для каждой удалённой записи.

## 3. Безопасные преобразования формата

Преобразование десятичной запятой — типичный пример проверяемой операции. Но мы сохраняем исходную строку. Для времени используем **явные два формата**, а не рискованную догадку о месяце/дне.

In [ ]:
def parse_number(value):
    s=str(value).replace(' ', '').replace('\u00a0','').replace(',','.').strip()
    try:
        return float(s) if s else float('nan')
    except ValueError:
        return float('nan')

def parse_time(value):
    s=str(value).strip()
    for fmt in ('%Y-%m-%d %H:%M','%d.%m.%Y %H:%M'):
        try:return datetime.strptime(s,fmt)
        except ValueError:pass
    return pd.NaT

probe = pd.DataFrame({'raw':['21,4','1 250','','нет данных'],'parsed':[parse_number(x) for x in ['21,4','1 250','','нет данных']]})
probe

In [ ]:
assert parse_number('21,4') == 21.4
assert parse_number('1 250') == 1250
assert pd.isna(parse_number(''))
assert parse_time('16.03.2026 10:00') == datetime(2026,3,16,10)
print('Четыре базовые проверки прошли')

## 4. Четыре решения для четырёх случаев

- `4200 ppm`: необычно, но не обязательно ошибочно; сохранить и отметить.
- `0 ppm`: недостоверное измерение в этой учебной ситуации; сохранить исходник и запросить уточнение.
- неизвестный код аудитории: не угадывать адрес автоматически.
- отсутствующий CO₂: использовать запись для других доступных признаков, не выдумывать показание.

**Упражнение:** напишите `make_quality_flags()` и создайте журнал минимум для 6 записей. Не подменяйте пропущенные значения числом 0.

In [ ]:
def demonstration_flag(co2_raw):
    value = parse_number(co2_raw)
    if pd.isna(value):return 'missing'
    if value <= 0:return 'invalid_requires_review'
    if value > 2000:return 'high_requires_review'
    return 'not_flagged'
for s in ['800','0','4200','']:
    print(s or '<пусто>', '->', demonstration_flag(s))

## 5. Объединить справочник и проверить кардинальность

Проверка `many_to_one` в pandas должна завершиться ошибкой при повторяющемся ключе в справочнике. Она не заменяет проверку неизвестных помещений.

In [ ]:
assert rooms.room_id.is_unique
known = raw.loc[raw.room_code.isin(rooms.room_id)].copy()
example = known.merge(rooms,left_on='room_code',right_on='room_id',how='left',validate='many_to_one')
assert len(known) == len(example)
print('Строк до/после соединения в демонстрации:',len(known),len(example))
print('Необработанных или неканонических кодов:',len(raw)-len(known))

## 6. Аудит чужого/ИИ-плана

> «Удалим все пропуски, выбросы CO₂ выше 2000 ppm и повторы числовых значений; затем сделаем `inner join` со справочником».

Напишите по одному контрпримеру к каждому решению и сформулируйте проверку. Затем выполняйте собственную лабораторную по HTML-заданию и сдайте отдельный результат.

In [ ]:
# Контрольная заготовка для вашего журнала: заполните реальные строки в своём решении.
log_columns=['record_id','issue','candidate_action','accepted_action','evidence','check']
student_log=pd.DataFrame(columns=log_columns)
print('Колонки журнала:',list(student_log.columns))

### После лабораторной

Проверьте, что можете объяснить: `число строк до = оставленные + полные копии + карантин`; почему экстремальное показание не нужно немедленно удалять; в каких случаях автоматическое исправление запрещено без первичного источника.